In [34]:
# ETS Model with Store Location and Holiday Flag, Validated on Test Set
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib.pyplot as plt

# Load data
train = pd.read_csv('./housing/train.csv')
test = pd.read_csv('./housing/test.csv')

In [35]:
# Display various statistics of the train dataset
print('Shape:', train.shape)
print('\nData types:')
print(train.dtypes)
print('\nMissing values:')
print(train.isnull().sum())
print('\nSummary statistics:')
print(train.describe(include='all'))

# Value counts for categorical columns
cat_cols = train.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    print(f'\nValue counts for {col}:')
    print(train[col].value_counts())

Shape: (1460, 81)

Data types:
Id                 int64
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
                  ...   
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
SalePrice          int64
Length: 81, dtype: object

Missing values:
Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

Summary statistics:
                 Id   MSSubClass MSZoning  LotFrontage        LotArea Street  \
count   1460.000000  1460.000000     1460  1201.000000    1460.000000   1460   
unique          NaN          NaN        5          NaN            NaN      2   
top             NaN          NaN       RL          NaN            NaN   Pave   
freq            NaN          NaN     1151          NaN

In [36]:
# Identify candidate columns for dichotomous variables, interactions, and polynomial terms
import numpy as np

# Dichotomous (binary) variable candidates: columns with exactly 2 unique values
binary_cols = [col for col in train.columns if train[col].nunique(dropna=False) == 2]
print('Candidate dichotomous (binary) columns:', binary_cols)

# Categorical columns with low cardinality (good for one-hot encoding or interactions)
cat_cols = train.select_dtypes(include=['object', 'category']).columns.tolist()
low_card_cat = [col for col in cat_cols if train[col].nunique() <= 10]
print('Low-cardinality categorical columns (<=10 unique values):', low_card_cat)

# Numeric columns for polynomial terms (exclude binary and ID columns)
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
poly_candidates = [col for col in numeric_cols if col not in binary_cols and 'id' not in col.lower() and train[col].nunique() > 10]
print('Numeric columns suitable for polynomial terms:', poly_candidates)

# Suggest possible interaction pairs (between low-cardinality categoricals and numerics)
interaction_pairs = [(cat, num) for cat in low_card_cat for num in poly_candidates]
print('Example interaction pairs (categorical x numeric):', interaction_pairs[:5], '...')

Candidate dichotomous (binary) columns: ['Street', 'Utilities', 'CentralAir']
Low-cardinality categorical columns (<=10 unique values): ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
Numeric columns suitable for polynomial terms: ['MSSubClass', 'LotFrontage', 'LotArea', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'TotRmsAbvGrd', 'GarageYrBlt', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch',

In [37]:
# Show columns that are candidates for dimension reduction (e.g., PCA)
# Criteria: numeric, not binary, not ID, not target, and with moderate/high cardinality
exclude_cols = set(binary_cols)
exclude_cols.update([col for col in train.columns if 'id' in col.lower()])
# Optionally exclude target column if known
target_col = 'target' if 'target' in train.columns else train.columns[-1]
exclude_cols.add(target_col)

# Candidates: numeric columns with >10 unique values, not in exclude_cols
candidates_dimred = [col for col in train.select_dtypes(include=[np.number]).columns if col not in exclude_cols and train[col].nunique() > 10]
print('Columns that are candidates for dimension reduction (e.g., PCA):', candidates_dimred)

Columns that are candidates for dimension reduction (e.g., PCA): ['MSSubClass', 'LotFrontage', 'LotArea', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'TotRmsAbvGrd', 'GarageYrBlt', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'MiscVal', 'MoSold']


In [38]:
# Handle NaN values: Option 1 - Fill NaN with default values (mean for numeric, mode for categorical)
for col in train.columns:
    if train[col].dtype in ['float64', 'int64']:
        train[col] = train[col].fillna(train[col].mean())
    else:
        train[col] = train[col].fillna(train[col].mode()[0])

for col in test.columns:
    if test[col].dtype in ['float64', 'int64']:
        test[col] = test[col].fillna(test[col].mean())
    else:
        test[col] = test[col].fillna(test[col].mode()[0])

# Option 2 (alternative): Drop rows with any NaN values
# train = train.dropna()
# test = test.dropna()

In [39]:
# --- Cell: Prepare training data (fit transformers, split, and transform) ---
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Identify target column
possible_targets = ['SalePrice']
target_col = None
for col in possible_targets:
    if col in train.columns:
        target_col = col
        break
if target_col is None:
    target_col = train.columns[-1]

# 2. Identify numeric and binary categorical columns (excluding target)
numeric_cols = [col for col in train.select_dtypes(include=[np.number]).columns if col != target_col]
binary_cat_cols = [col for col in train.select_dtypes(include=['object', 'category']).columns if train[col].nunique() == 2]

# 3. Identify columns for PCA (excluding target)
candidates_dimred = [col for col in candidates_dimred if col != target_col]

# 4. Prepare target
y = train[target_col]

# 5. 80/20 split (do this before fitting transformers to avoid leakage)
X_numeric = train[numeric_cols]
X_bin_df = train[binary_cat_cols] if binary_cat_cols else pd.DataFrame(index=train.index)
X_pca_df = train[candidates_dimred] if candidates_dimred else pd.DataFrame(index=train.index)

X_train_num, X_val_num, X_train_bin, X_val_bin, X_train_pca, X_val_pca, y_train, y_val = train_test_split(
    X_numeric, X_bin_df, X_pca_df, y, test_size=0.2, random_state=42
)

# 6. Scale numeric columns (fit only on train)
scaler = StandardScaler()
X_train_numeric_scaled = scaler.fit_transform(X_train_num)
X_val_numeric_scaled = scaler.transform(X_val_num)

# 7. Binary encode binary categorical columns (fit only on train)
binarizers = {}
X_train_bin_encoded = []
X_val_bin_encoded = []
for col in X_train_bin.columns:
    lb = LabelBinarizer()
    lb.fit(X_train_bin[col].astype(str))
    binarizers[col] = lb
    train_encoded = lb.transform(X_train_bin[col].astype(str))
    val_encoded = lb.transform(X_val_bin[col].astype(str))
    if train_encoded.ndim == 1:
        train_encoded = train_encoded.reshape(-1, 1)
        val_encoded = val_encoded.reshape(-1, 1)
    X_train_bin_encoded.append(train_encoded)
    X_val_bin_encoded.append(val_encoded)
if X_train_bin_encoded:
    X_train_bin_encoded = np.hstack(X_train_bin_encoded)
    X_val_bin_encoded = np.hstack(X_val_bin_encoded)
else:
    X_train_bin_encoded = np.empty((len(X_train_bin), 0))
    X_val_bin_encoded = np.empty((len(X_val_bin), 0))

# 8. PCA on highly correlated columns (fit only on train)
if X_train_pca.shape[1] > 0:
    pca = PCA(n_components=0.95, random_state=42)
    X_train_pca_trans = pca.fit_transform(X_train_pca)
    X_val_pca_trans = pca.transform(X_val_pca)
else:
    pca = None
    X_train_pca_trans = np.empty((len(X_train_pca), 0))
    X_val_pca_trans = np.empty((len(X_val_pca), 0))

# 9. Combine all features
from numpy import hstack
X_train_all = hstack([X_train_numeric_scaled, X_train_bin_encoded, X_train_pca_trans])
X_val_all = hstack([X_val_numeric_scaled, X_val_bin_encoded, X_val_pca_trans])

# 10. Store all fitted objects and splits for reuse
prepared_data = {
    'X_train_all': X_train_all,
    'X_val_all': X_val_all,
    'y_train': y_train,
    'y_val': y_val,
    'scaler': scaler,
    'binarizers': binarizers,
    'pca': pca,
    'numeric_cols': numeric_cols,
    'binary_cat_cols': binary_cat_cols,
    'candidates_dimred': candidates_dimred,
    'target_col': target_col
}
print('Training data prepared. Use prepared_data for model training and validation.')

Training data prepared. Use prepared_data for model training and validation.


In [40]:
# Prepare all test data for prediction (no split, same as train prep but using fitted transformers)
import numpy as np

# Use column lists and fitted transformers from prepared_data
test_numeric = test[prepared_data['numeric_cols']]
test_numeric_scaled = prepared_data['scaler'].transform(test_numeric)

# Binary encode binary categorical columns using fitted binarizers
X_test_bin_encoded = []
for col in prepared_data['binary_cat_cols']:
    lb = prepared_data['binarizers'][col]
    test_encoded = lb.transform(test[col].astype(str))
    if test_encoded.ndim == 1:
        test_encoded = test_encoded.reshape(-1, 1)
    X_test_bin_encoded.append(test_encoded)
if X_test_bin_encoded:
    X_test_bin_encoded = np.hstack(X_test_bin_encoded)
else:
    X_test_bin_encoded = np.empty((len(test), 0))

# PCA on highly correlated columns using fitted PCA
if prepared_data['candidates_dimred'] and prepared_data['pca'] is not None:
    test_pca = test[prepared_data['candidates_dimred']]
    test_pca_trans = prepared_data['pca'].transform(test_pca)
else:
    test_pca_trans = np.empty((len(test), 0))

# Combine all features
X_test_all = np.hstack([test_numeric_scaled, X_test_bin_encoded, test_pca_trans])
print('All test data prepared. Use X_test_all for predictions.')

All test data prepared. Use X_test_all for predictions.


In [41]:
# Train linear regression model and print validation metrics
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Use prepared_data from training data prep cell
X_train = prepared_data['X_train_all']
X_val = prepared_data['X_val_all']
y_train = prepared_data['y_train']
y_val = prepared_data['y_val']

reg = LinearRegression()
reg.fit(X_train, y_train)
y_pred = reg.predict(X_val)

# Metrics
mae = mean_absolute_error(y_val, y_pred)
me = np.mean(y_val - y_pred)
mape = np.mean(np.abs((y_val - y_pred) / y_val)) * 100
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

print(f'MAE: {mae:.2f}')
print(f'ME: {me:.2f}')
print(f'MAPE: {mape:.2f}%')
print(f'RMSE: {rmse:.2f}')
print(f'MSE: {mse:.2f}')
print(f'R2 Score: {r2:.4f}')

MAE: 22848.82
ME: 4189.23
MAPE: 13.33%
RMSE: 36671.55
MSE: 1344802916.44
R2 Score: 0.8247


NameError: name 'feature_names' is not defined

In [ ]:
# Predict SalePrice for test.csv using the trained regression model and prepared test data
# Assumes X_test_all is prepared and reg is the trained model from previous cell

test_ids = test['Id'] if 'Id' in test.columns else test.index
saleprice_pred = reg.predict(X_test_all)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': saleprice_pred})
submission.to_csv('./store/submission_linear.csv', index=False)
print('Predictions saved to ./store/submission_linear.csv')

Predictions saved to submission_linear.csv


In [ ]:
# Train Ridge regression model and print validation metrics
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X_train = prepared_data['X_train_all']
X_val = prepared_data['X_val_all']
y_train = prepared_data['y_train']
y_val = prepared_data['y_val']

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred = ridge.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
me = np.mean(y_val - y_pred)
mape = np.mean(np.abs((y_val - y_pred) / y_val)) * 100
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

print(f'Ridge Regression:')
print(f'MAE: {mae:.2f}')
print(f'ME: {me:.2f}')
print(f'MAPE: {mape:.2f}%')
print(f'RMSE: {rmse:.2f}')
print(f'MSE: {mse:.2f}')
print(f'R2 Score: {r2:.4f}')

Ridge Regression:
MAE: 21575.72
ME: 3258.43
MAPE: 13.00%
RMSE: 35316.44
MSE: 1247251090.36
R2 Score: 0.8374


In [ ]:
# Predict SalePrice for test.csv using the trained regression model and prepared test data
# Assumes X_test_all is prepared and reg is the trained model from previous cell

test_ids = test['Id'] if 'Id' in test.columns else test.index
saleprice_pred = ridge.predict(X_test_all)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': saleprice_pred})
submission.to_csv('./store/submission_ridge.csv', index=False)
print('Predictions saved to ./store/submission_ridge.csv')

Predictions saved to ./store/submission_ridge.csv


In [ ]:
# Train Lasso regression model and print validation metrics
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X_train = prepared_data['X_train_all']
X_val = prepared_data['X_val_all']
y_train = prepared_data['y_train']
y_val = prepared_data['y_val']

lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
me = np.mean(y_val - y_pred)
mape = np.mean(np.abs((y_val - y_pred) / y_val)) * 100
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

print(f'Lasso Regression:')
print(f'MAE: {mae:.2f}')
print(f'ME: {me:.2f}')
print(f'MAPE: {mape:.2f}%')
print(f'RMSE: {rmse:.2f}')
print(f'MSE: {mse:.2f}')
print(f'R2 Score: {r2:.4f}')

Lasso Regression:
MAE: 21585.24
ME: 3261.44
MAPE: 13.01%
RMSE: 35327.65
MSE: 1248042778.32
R2 Score: 0.8373


C:\Users\anujb\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.882e+10, tolerance: 6.967e+08
  model = cd_fast.enet_coordinate_descent(


In [ ]:
# Train XGBoost regression model and print validation metrics
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X_train = prepared_data['X_train_all']
X_val = prepared_data['X_val_all']
y_train = prepared_data['y_train']
y_val = prepared_data['y_val']

xgb = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
me = np.mean(y_val - y_pred)
mape = np.mean(np.abs((y_val - y_pred) / y_val)) * 100
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

print(f'XGBoost Regression:')
print(f'MAE: {mae:.2f}')
print(f'ME: {me:.2f}')
print(f'MAPE: {mape:.2f}%')
print(f'RMSE: {rmse:.2f}')
print(f'MSE: {mse:.2f}')
print(f'R2 Score: {r2:.4f}')

XGBoost Regression:
MAE: 16155.41
ME: 491.79
MAPE: 9.82%
RMSE: 25107.73
MSE: 630397952.00
R2 Score: 0.9178


In [ ]:
# Predict SalePrice for test.csv using the trained regression model and prepared test data
# Assumes X_test_all is prepared and reg is the trained model from previous cell

test_ids = test['Id'] if 'Id' in test.columns else test.index
saleprice_pred = xgb.predict(X_test_all)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': saleprice_pred})
submission.to_csv('./store/submission_xgb.csv', index=False)
print('Predictions saved to ./store/submission_xgb.csv')

Predictions saved to ./store/submission_xgb.csv


In [45]:
# Display feature importances or coefficients for all models
import numpy as np

print('--- Linear Regression Coefficients ---')
if hasattr(reg, 'coef_'):
    print('Linear Regression:')
    print(reg.coef_)

print('\n--- XGBoost Feature Importances ---')
if hasattr(xgb, 'feature_importances_'):
    print('XGBoost Feature Importances:')
    print(xgb.feature_importances_)



--- Linear Regression Coefficients ---
Linear Regression:
[-9.41129370e+02 -8.59028745e+03 -3.12494390e+03 -1.50881914e+01
  2.44241541e+04  5.28455841e+03  9.51381422e+03  3.50059102e+03
  4.32119404e+03  3.68031086e+03 -5.08896659e+02 -6.80592561e+02
  2.96184538e+03  7.62686868e+03  8.92445078e+03  5.36495104e+02
  1.31527677e+04  5.92598443e+03 -1.03625067e+02  1.29101346e+03
 -1.10597309e+03 -6.63526458e+03 -2.39571222e+03  8.36022855e+03
  3.37467979e+03  3.17997419e+03  9.05562024e+03 -5.96615848e+02
  2.82988169e+03 -4.84170422e+02  6.05911545e+02  1.27532472e+03
  3.83332931e+03 -7.72199986e+02 -3.70870070e+02 -4.78718661e+02
 -6.99664346e+02  1.43628319e+04 -3.54883354e+02 -5.12922061e+04
 -8.96926371e+03  4.78362552e-01]

--- XGBoost Feature Importances ---
XGBoost Feature Importances:
[6.82807877e-04 2.10388913e-03 3.48973414e-03 3.62674706e-03
 5.32643497e-03 0.00000000e+00 0.00000000e+00 2.15961132e-03
 1.15004275e-02 0.00000000e+00 1.21360389e-03 0.00000000e+00
 1.043407

In [46]:
# Print top 5 feature names for linear regression based on highest absolute coefficients
if hasattr(reg, 'coef_') and 'numeric_cols' in prepared_data:
    feature_names = prepared_data['numeric_cols'][:]
    if prepared_data['binary_cat_cols']:
        feature_names += prepared_data['binary_cat_cols']
    if prepared_data['candidates_dimred'] and prepared_data['pca'] is not None:
        feature_names += [f'PCA_{i+1}' for i in range(prepared_data['pca'].n_components_)]
    coefs = reg.coef_
    # Get indices of top 5 absolute coefficients
    top5_idx = np.argsort(np.abs(coefs))[::-1][:5]
    print('Top 5 features by absolute coefficient (Linear Regression):')
    for idx in top5_idx:
        print(f'{feature_names[idx]}: {coefs[idx]:.4f}')
else:
    print('Linear regression coefficients or feature names not available.')

Top 5 features by absolute coefficient (Linear Regression):
Utilities: -51292.2061
OverallQual: 24424.1541
Street: 14362.8319
GrLivArea: 13152.7677
YearBuilt: 9513.8142
